In [ ]:
pip install biopython

IMPORTANTE

O alinhamento múltiplo foi realizado utilizando o Clustal.

No entanto, esta ferramenta funciona corretamente no Google Colab, mas não no Jupyter Notebook do Visual Studio Code.

Por esse motivo, é disponibilizado em seguida um link para um notebook do Google Colab, que contém todo o código desenvolvido neste trabalho.

ALINHAMENTO MULTIPLO

Moduluos Necessarios

In [ ]:
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

import subprocess

In [16]:
record = SeqIO.read("AAT1_genome.gb", "genbank")
gene_sequences = {}
genes = [
    {"locus": "P9A56_gp33", "name": "endolysin"},
    {"locus": "P9A56_gp34", "name": "putative i-spanin"},
    {"locus": "P9A56_gp35", "name": "putative o-spanin"}
]

for gene in genes:
    gene_locus = gene["locus"]
    for feature in record.features:
        if feature.type == "CDS" and feature.qualifiers.get("locus_tag", [""])[0] == gene_locus:
            protein_seq = feature.qualifiers["translation"][0]
            gene_sequences[gene_locus] = protein_seq

homologos = [
    ("hit1", "SEQUENCIA_DO_HOMOLOGO1"),
    ("hit2", "SEQUENCIA_DO_HOMOLOGO2")
]

for gene_id, seq in gene_sequences.items():
    homologs = [(gene_id, seq)] + homologos
    records = [SeqRecord(Seq(s), id=name, description="") for name, s in homologs]
    SeqIO.write(records, f"{gene_id}_homologs.fasta", "fasta")

Tentamos usar o codigo 1 mas não deu, por isso usamos o codigo na celula a baixo

1- #from Bio.Align.Applications import ClustalwCommandline ---> Não esta a funcionar por isso tentamos desta forma

In [20]:
!apt-get update
!apt-get install -y clustalo

'apt-get' is not recognized as an internal or external command,
operable program or batch file.
'apt-get' is not recognized as an internal or external command,
operable program or batch file.


As sequencias alinhadas com os seus homologos

In [ ]:
fasta_files = [
    "P9A56_gp33_homologs.fasta",
    "P9A56_gp34_homologs.fasta",
    "P9A56_gp35_homologs.fasta"
]

for fasta_file in fasta_files:
    aligned_file = fasta_file.replace("_homologs.fasta", "_aligned.fasta")

    cmd = [
        "clustalo",
        "-i", fasta_file,
        "-o", aligned_file,
        "--force",
        "--verbose"
    ]

    subprocess.run(cmd, check=True)

    print(f"Alinhamento múltiplo gerado: {aligned_file}")

In [ ]:
from Bio import AlignIO 
#Ler os Alinhamentos
aligned_files = [
    "P9A56_gp33_aligned.fasta",
    "P9A56_gp34_aligned.fasta",
    "P9A56_gp35_aligned.fasta"
]

for alinhamento in aligned_files:
  alignment = AlignIO.read(alinhamento, "fasta") 
  print(alignment )

Tudo alinhado

In [ ]:
aligned_files = [
    "P9A56_gp33_homologs.fasta",
    "P9A56_gp34_homologs.fasta",
    "P9A56_gp35_homologs.fasta"
]

# Criar um arquivo combinado
combined_file = "combined_genes.fasta"
with open(combined_file, "w") as outfile:
    for f in aligned_files:
        for record in SeqIO.parse(f, "fasta"):
            SeqIO.write(record, outfile, "fasta")

import subprocess

everything_aligned = "combined_genes_aligned.fasta"

cmd = [
    "clustalo",           # Executável Clustal Omega
    "-i", combined_file,  # arquivo de entrada
    "-o", everything_aligned,   # arquivo de saída
    "--force",            # sobrescrever se existir
    "--verbose"
]

subprocess.run(cmd, check=True)
print(f"Alinhamento múltiplo gerado: {everything_aligned}")

everything_alin = AlignIO.read(everything_aligned, "fasta")
print(everything_alin)

ARVORES FILOGENETICAS

Modulos Necessarios

In [ ]:
from Bio import AlignIO
from Bio.Phylo.TreeConstruction import DistanceCalculator, DistanceTreeConstructor
from Bio import Phylo
from Bio.Align import MultipleSeqAlignment

Arvores das sequencias alinhadas com os seus homologos

In [ ]:
aligned_files = [
    "P9A56_gp33_aligned.fasta",
    "P9A56_gp34_aligned.fasta",
    "P9A56_gp35_aligned.fasta"
]

for aligned_file in aligned_files:
    alignment = AlignIO.read(aligned_file, "fasta")


    calculator = DistanceCalculator('identity')
    dm = calculator.get_distance(alignment)

    constructor = DistanceTreeConstructor()
    tree = constructor.nj(dm)


    tree_file = aligned_file.replace("_aligned.fasta", "_tree.xml")
    Phylo.write(tree, tree_file, "phyloxml")


    print(f"Árvore filogenética gerada: {tree_file}")
    Phylo.draw(tree)

Arvore das tres sequencias

In [ ]:
everything_alin = AlignIO.read(everything_aligned, "fasta")

seqs_alinhadas = [everything_alin[0],
                  everything_alin[3],
                  everything_alin[6]]

novo_alinhamento = MultipleSeqAlignment(seqs_alinhadas)

calculator = DistanceCalculator('identity')
distance_matrix = calculator.get_distance(novo_alinhamento)

constructor = DistanceTreeConstructor()
tree = constructor.upgma(distance_matrix)

Phylo.draw_ascii(tree)
